In [1]:
prefix_path = '../..'

import sys
sys.path.append(prefix_path)

import json
import pandas as pd
import numpy as np

In [2]:
DATASET_PATH = f'{prefix_path}/data/phd/phd.json'
SAVE_PATH = f'{prefix_path}/data/phd/phd_base.json'

#### Construct Base PhD dataset

In [3]:
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    dataset = json.load(f)
df = pd.DataFrame(dataset)

# Skip css data
df = df[df['ccs_description'].isna()]
# Convert NaN to null
df = df.replace({np.nan: None})

print(f'PhD dataset size: {len(df)}')
df.head()

PhD dataset size: 16844


,task,yes_question,no_question,context,image_id,hitem,subject,gt,ccs_description
0,counting,Are there three couches in the image?,Are there 2 couches in the image?,"{'icc': 'In a cozy living room setting, two co...",000000262207,2,couches,three,None
1,sentiment,Is the cat looking sad in the image?,Is the cat content in the image?,{'icc': 'The cat in the image appears to be co...,000000354493,content,cat,sad,None
2,counting,Are there four people in the image?,Are there exactly three people in the image?,"{'icc': 'In the image, there are exactly three...",000000102532,3,people,4,None
3,sentiment,Is the boy appearing happy in the image?,Is the boy displaying a mischievous expression...,"{'icc': 'In a delightful scene, the boy is dis...",000000099810,mischievous,boy,happy,None
4,positional,Is there an object located two hundred and thi...,Is there anything in front of the bike in the ...,{'icc': 'In the thrilling moment captured in t...,000000049683,0,front of the bike,two hundred and thirteen,None


In [4]:
df = df.reset_index(drop=True)
df = df.assign(couple_id=df.index)

df_yes_question = df.assign(
    question=df['yes_question'],
    question_gt=1,
)[["couple_id", "task", "hitem", "subject", "gt", "question", "image_id", "question_gt"]]

df_no_question = df.assign(
    question=df['no_question'],
    question_gt=0,
)[["couple_id", "task", "hitem", "subject", "gt", "question", "image_id", "question_gt"]]

df_base = pd.concat([df_yes_question, df_no_question], ignore_index=True)
# Clean up
df_base = df_base.sample(frac=1, random_state=42).reset_index(drop=True)
df_base = df_base.reset_index().rename(columns={'index': 'id'})
df_base['id'] = df_base['id'].astype(str)
df_base['couple_id'] = df_base['couple_id'].astype(str)

print("Base PhD dataset size:", len(df_base))
df_base.head()

Base PhD dataset size: 33688


,id,couple_id,task,hitem,subject,gt,question,image_id,question_gt
0,0,10961,sentiment,nostalgic,people,happy,Are the people in the image feeling nostalgic?,000000039538,0
1,1,8598,object,helmet,None,cat,Is there a helmet in the image?,000000116031,0
2,2,2995,object,dishwasher,None,oven,Is there a dishwasher in the image?,000000470113,0
3,3,2678,object,ostrich,None,giraffe,Is there a giraffe in the image?,000000569718,1
4,4,12285,counting,1,toothbrushes,5,Are there five toothbrushes in the image?,000000388376,1


In [5]:
dataset_base = df_base.to_dict(orient="records")

with open(SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(dataset_base, f, indent=2)